# StarFire Language Model Training

**IMPORTANT**: This notebook trains the model. The final model MUST work in pure Rust.

This is for GPU-accelerated training only. Weights are exported in binary format
compatible with Rust's `CharRNN::load()`.

## Workflow
1. Train in Colab (GPU)
2. Download `model.bin` to `data/star_model.bin`
3. Load in Rust with `cargo run --bin test_save_load`


## Setup

Mount Google Drive and install dependencies.

In [1]:
import os
import torch

# Secrets from environment
HF_TOKEN = os.environ.get('HF_TOKEN', '')
WANDB_API_KEY = os.environ.get('WANDB_API_KEY', '')

print(f"HuggingFace token set: {bool(HF_TOKEN)}")
print(f"WandB token set: {bool(WANDB_API_KEY)}")

# Login to WandB
if WANDB_API_KEY:
    !wandb login $WANDB_API_KEY

# Config - MUST match Rust defaults
class C:
    VOCAB_SIZE = 227
    EMBEDDING_DIM = 64
    HIDDEN_SIZE = 256
    NUM_LAYERS = 2
    DROPOUT = 0.1
    SEQ_LENGTH = 128
    BATCH_SIZE = 32
    LEARNING_RATE = 0.001
    GRADIENT_CLIP = 5.0
    EPOCHS = 50
    # Google Drive paths - upload all_personal_training.txt to /content/drive/MyDrive/starfire/
    CHECKPOINT_DIR = '/content/drive/MyDrive/starfire/checkpoints'
    DATA_PATH = '/content/drive/MyDrive/starfire/all_personal_training.txt'
    LOG_INTERVAL = 50
    SAVE_INTERVAL = 500
    WANDB_PROJECT = 'starfire-lm'
    HF_REPO_ID = None

config = C()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
os.makedirs(config.CHECKPOINT_DIR, exist_ok=True)


Device: cuda


## Config

Set hyperparameters and authenticate.

In [2]:
import os
import torch

# Secrets from environment
HF_TOKEN = os.environ.get('HF_TOKEN', '')
WANDB_API_KEY = os.environ.get('WANDB_API_KEY', '')

print(f"HuggingFace token set: {bool(HF_TOKEN)}")
print(f"WandB token set: {bool(WANDB_API_KEY)}")

# Login to WandB
if WANDB_API_KEY:
    !wandb login $WANDB_API_KEY

# Config - MUST match Rust defaults
class C:
    VOCAB_SIZE = 227
    EMBEDDING_DIM = 64
    HIDDEN_SIZE = 256
    NUM_LAYERS = 2
    DROPOUT = 0.1
    SEQ_LENGTH = 128
    BATCH_SIZE = 32
    LEARNING_RATE = 0.001
    GRADIENT_CLIP = 5.0
    EPOCHS = 50
    # Local data file - update if needed
    CHECKPOINT_DIR = './checkpoints'
    DATA_PATH = './all_personal_training.txt'
    LOG_INTERVAL = 50
    SAVE_INTERVAL = 500
    WANDB_PROJECT = 'starfire-lm'
    HF_REPO_ID = None

config = C()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

HuggingFace token set: False
WandB token set: False
Device: cuda


## Vocabulary

Must match `lib/language_model/vocabulary.rs` exactly:
- pad=0, eos=1
- ASCII 32-126
- Extended 128-255

In [3]:
class Vocabulary:
    def __init__(self):
        self.pad = 0
        self.eos = 1
        self.special_count = 2

        self.chars = []
        for i in range(32, 127):
            self.chars.append(chr(i))
        for i in range(128, 256):
            self.chars.append(chr(i))

        self.char_to_idx = {c: i + 2 for i, c in enumerate(self.chars)}
        self.idx_to_char = {i + 2: c for i, c in enumerate(self.chars)}
        self.pad_idx = 0
        self.eos_idx = 1

    @property
    def size(self):
        return len(self.chars) + 2

    def encode(self, text):
        return [self.char_to_idx.get(c, 0) for c in text]

    def decode(self, indices):
        result = []
        for idx in indices:
            if idx == 1:
                break
            if idx >= 2:
                result.append(self.idx_to_char.get(idx, ''))
        return ''.join(result)

    def encode_with_eos(self, text):
        return self.encode(text) + [1]

vocab = Vocabulary()
print(f"Vocab size: {vocab.size}")

Vocab size: 225


## Model

PyTorch model with architecture matching Rust `CharRNN`.

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class LSTMCellPy(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        total = input_size + hidden_size
        scale = math.sqrt(2.0 / total)

        self.i_weight = nn.Parameter(torch.randn(hidden_size, total) * scale)
        self.f_weight = nn.Parameter(torch.randn(hidden_size, total) * scale)
        self.c_weight = nn.Parameter(torch.randn(hidden_size, total) * scale)
        self.o_weight = nn.Parameter(torch.randn(hidden_size, total) * scale)

        self.i_bias = nn.Parameter(torch.zeros(hidden_size))
        self.f_bias = nn.Parameter(torch.ones(hidden_size))
        self.c_bias = nn.Parameter(torch.zeros(hidden_size))
        self.o_bias = nn.Parameter(torch.zeros(hidden_size))

    def forward(self, x, h, c):
        combined = torch.cat([x, h], dim=-1)
        i = torch.sigmoid(F.linear(combined, self.i_weight, self.i_bias))
        f = torch.sigmoid(F.linear(combined, self.f_weight, self.f_bias))
        c_tilde = torch.tanh(F.linear(combined, self.c_weight, self.c_bias))
        o = torch.sigmoid(F.linear(combined, self.o_weight, self.o_bias))
        c_new = f * c + i * c_tilde
        h_new = o * torch.tanh(c_new)
        return h_new, c_new

class CharLSTM(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.embedding = nn.Embedding(config.VOCAB_SIZE, config.EMBEDDING_DIM)
        self.lstm_cells = nn.ModuleList()
        for layer in range(config.NUM_LAYERS):
            input_size = config.EMBEDDING_DIM if layer == 0 else config.HIDDEN_SIZE
            self.lstm_cells.append(LSTMCellPy(input_size, config.HIDDEN_SIZE))
        self.output_weight = nn.Parameter(torch.randn(config.VOCAB_SIZE, config.HIDDEN_SIZE) * 0.1)
        self.output_bias = nn.Parameter(torch.zeros(config.VOCAB_SIZE))

    def forward(self, x, hidden=None):
        batch_size, seq_len = x.shape
        if hidden is None:
            h = [torch.zeros(batch_size, self.config.HIDDEN_SIZE, device=x.device)
                 for _ in range(self.config.NUM_LAYERS)]
            c = [torch.zeros(batch_size, self.config.HIDDEN_SIZE, device=x.device)
                 for _ in range(self.config.NUM_LAYERS)]
        else:
            h, c = hidden

        x = self.embedding(x)
        all_logits = []

        for t in range(seq_len):
            input_t = x[:, t, :]
            for layer_idx in range(self.config.NUM_LAYERS):
                h[layer_idx], c[layer_idx] = self.lstm_cells[layer_idx](input_t, h[layer_idx], c[layer_idx])
                input_t = h[layer_idx]
            logits_t = F.linear(input_t, self.output_weight, self.output_bias)
            all_logits.append(logits_t)

        return torch.stack(all_logits, dim=1), (h, c)

    @property
    def num_params(self):
        return sum(p.numel() for p in self.parameters())

model = CharLSTM(config).to(device)
print(f"Parameters: {model.num_params:,}")

Parameters: 926,883


## Dataset

Load training data from Google Drive.

In [5]:
# Dataset - Load training data from Google Drive
import os
from torch.utils.data import Dataset, DataLoader

class Vocabulary:
    def __init__(self):
        self.pad = 0
        self.eos = 1
        self.special_count = 2
        self.chars = []
        for i in range(32, 127):
            self.chars.append(chr(i))
        for i in range(128, 256):
            self.chars.append(chr(i))
        self.char_to_idx = {c: i + 2 for i, c in enumerate(self.chars)}
        self.idx_to_char = {i + 2: c for i, c in enumerate(self.chars)}
        self.pad_idx = 0
        self.eos_idx = 1

    @property
    def size(self):
        return len(self.chars) + 2

    def encode(self, text):
        return [self.char_to_idx.get(c, 0) for c in text]

    def decode(self, indices):
        result = []
        for idx in indices:
            if idx == 1:
                break
            if idx >= 2:
                result.append(self.idx_to_char.get(idx, ''))
        return ''.join(result)

    def encode_with_eos(self, text):
        return self.encode(text) + [1]

class ConvDataset(Dataset):
    def __init__(self, path, vocab, seq_len):
        self.vocab = vocab
        self.seq_len = seq_len
        self.sequences = self._load(path)

    def _load(self, path):
        sequences = []
        if not os.path.exists(path):
            raise FileNotFoundError(f"Training data not found at {path}. Please upload all_personal_training.txt to Google Drive at /content/drive/MyDrive/starfire/")
        with open(path, 'r', encoding='utf-8', errors='ignore') as f:
            content = f.read()
        if not content.strip():
            raise ValueError(f"Training data at {path} is empty.")
        content = content.replace('user:', 'Zachary:').replace('assistant:', 'Star:')
        lines = content.split('\n')
        current = []
        for line in lines:
            line = line.strip()
            if not line:
                if current:
                    text = '\n'.join(current) + '\n'
                    enc = self.vocab.encode_with_eos(text)
                    if len(enc) >= 10:
                        sequences.append(enc)
                    current = []
            else:
                current.append(line)
        if current:
            text = '\n'.join(current) + '\n'
            enc = self.vocab.encode_with_eos(text)
            if len(enc) >= 10:
                sequences.append(enc)
        return sequences

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        seq = self.sequences[idx]
        if len(seq) > self.seq_len:
            start = max(0, len(seq) - self.seq_len)
            seq = seq[start:start + self.seq_len]
        else:
            seq = seq + [0] * (self.seq_len - len(seq))
        return torch.tensor(seq[:-1]), torch.tensor(seq[1:])

vocab = Vocabulary()
print(f"Vocab size: {vocab.size}")
dataset = ConvDataset(config.DATA_PATH, vocab, config.SEQ_LENGTH)
print(f"Sequences: {len(dataset)}")
dataloader = DataLoader(dataset, batch_size=config.BATCH_SIZE, shuffle=True, num_workers=0)


Vocab size: 225


FileNotFoundError: Training data not found at ./all_personal_training.txt

## Training

Train with BPTT via PyTorch autograd.

In [ ]:
import wandb
from tqdm.notebook import tqdm

def compute_loss(logits, targets, pad_idx):
    batch_size, seq_len, vocab_size = logits.shape
    logits_flat = logits.reshape(-1, vocab_size)
    targets_flat = targets.reshape(-1)
    mask = (targets_flat != pad_idx).float()
    loss = F.cross_entropy(logits_flat, targets_flat, reduction='none')
    return (loss * mask).sum() / mask.sum()

def train_epoch(model, dataloader, optimizer, config, epoch, wandb_run=None):
    model.train()
    total_loss, num_batches = 0, 0
    pbar = tqdm(dataloader, desc=f"Epoch {epoch}")
    hidden = None

    for batch_idx, (x, y) in enumerate(pbar):
        x, y = x.to(device), y.to(device)

        if hidden is not None:
            hidden = ([h.detach() for h in hidden[0]],
                     [c.detach() for c in hidden[1]])

        logits, hidden = model(x, hidden)
        loss = compute_loss(logits, y, vocab.pad_idx)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), config.GRADIENT_CLIP)
        optimizer.step()

        total_loss += loss.item()
        num_batches += 1

        if batch_idx % config.LOG_INTERVAL == 0:
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})
            if wandb_run:
                wandb_run.log({'loss': loss.item(), 'epoch': epoch, 'batch': batch_idx})

        if batch_idx > 0 and batch_idx % config.SAVE_INTERVAL == 0:
            save_checkpoint(model, optimizer, epoch, batch_idx, config)

    return total_loss / num_batches

def save_checkpoint(model, optimizer, epoch, batch_idx, config):
    os.makedirs(config.CHECKPOINT_DIR, exist_ok=True)
    path = os.path.join(config.CHECKPOINT_DIR, f'ckpt_e{epoch}_b{batch_idx}.pt')
    torch.save({
        'epoch': epoch, 'batch_idx': batch_idx,
        'model': model.state_dict(),
        'optimizer': optimizer.state_dict()
    }, path)
    print(f"\nSaved: {path}")

In [ ]:
wandb_run = None  # Disabled
# if WANDB_API_KEY:
#     wandb_run = wandb.init(project=config.WANDB_PROJECT, config=config.__dict__)

optimizer = torch.optim.Adam(model.parameters(), lr=config.LEARNING_RATE)

for epoch in range(1, config.EPOCHS + 1):
    avg_loss = train_epoch(model, dataloader, optimizer, config, epoch, wandb_run)
    print(f"Epoch {epoch} done. Loss: {avg_loss:.4f}")
    save_checkpoint(model, optimizer, epoch, 0, config)

if wandb_run:
    wandb_run.finish()

## Generate

Test text generation.

In [ ]:
def generate(model, vocab, prompt, max_len=200, temp=1.0, seed=None):
    model.eval()
    if seed:
        torch.manual_seed(seed)

    with torch.no_grad():
        ids = vocab.encode(prompt)
        tensor = torch.tensor([ids], dtype=torch.long).to(device)
        hidden = None
        generated = ids

        for _ in range(max_len):
            logits, hidden = model(tensor, hidden)
            last = logits[0, -1, :]
            if temp != 1.0:
                last = last / temp
            probs = F.softmax(last, dim=-1)
            next_tok = torch.multinomial(probs, 1).item()
            if next_tok == 1:
                break
            generated.append(next_tok)
            tensor = torch.tensor([[next_tok]], dtype=torch.long).to(device)

        return vocab.decode(generated)

print(generate(model, vocab, "Hello", max_len=100, temp=0.8, seed=42))

## Export to Rust Binary

**CRITICAL**: Save in exact Rust format.

Download `model.bin` and place at `data/star_model.bin`.

In [ ]:
import struct

def save_binary(model, path):
    with open(path, 'wb') as f:
        def w_u32(v): f.write(struct.pack('<I', v))
        def w_f32(v): f.write(struct.pack('<f', v))
        def w_vec(v):
            f.write(struct.pack('<Q', len(v)))
            for x in v: f.write(struct.pack('<f', x))

        c = model.config
        w_u32(c.VOCAB_SIZE)
        w_u32(c.EMBEDDING_DIM)
        w_u32(c.HIDDEN_SIZE)
        w_u32(c.NUM_LAYERS)
        w_f32(c.DROPOUT)

        w_vec(model.embedding.weight.data.cpu().numpy().flatten().tolist())

        for cell in model.lstm_cells:
            w_vec(cell.i_weight.data.cpu().numpy().flatten().tolist())
            w_vec(cell.i_bias.data.cpu().numpy().tolist())
            w_vec(cell.f_weight.data.cpu().numpy().flatten().tolist())
            w_vec(cell.f_bias.data.cpu().numpy().tolist())
            w_vec(cell.c_weight.data.cpu().numpy().flatten().tolist())
            w_vec(cell.c_bias.data.cpu().numpy().tolist())
            w_vec(cell.o_weight.data.cpu().numpy().flatten().tolist())
            w_vec(cell.o_bias.data.cpu().numpy().tolist())

        w_vec(model.output_weight.data.cpu().numpy().flatten().tolist())
        w_vec(model.output_bias.data.cpu().numpy().tolist())

    size_mb = os.path.getsize(path) / 1024 / 1024
    print(f"Saved: {path} ({size_mb:.2f} MB)")

os.makedirs(config.CHECKPOINT_DIR, exist_ok=True)
binary_path = os.path.join(config.CHECKPOINT_DIR, 'model.bin')
save_binary(model, binary_path)
print("\nCopy model.bin to data/star_model.bin for Rust to use")